# C3: Bagging (Bootstrap Aggregating) Ensembles

## Introduction to Ensemble Learning

In machine learning, **Ensemble Learning** is the process of combining multiple models (often called "weak learners") to create a single, more powerful model (a "strong learner"). The idea is based on the "Wisdom of Crowds": a group of average predictors can often make better decisions than a single expert predictor.

### Bagging (Bootstrap Aggregating)

**Bagging** involves training multiple base models (usually Decision Trees) in parallel. Each Model is trained on a random subset of the data.

It consists of two main steps:
1.  **Bootstrapping**: Creating random training datasets by sampling with replacement from the original data.
2.  **Aggregation**: Combining the predictions from all models.
    -   **Classification**: Majority Voting (Soft or Hard).
    -   **Regression**: Averaging.

This technique helps reduce **Variance** (overfitting).

![Bagging Architecture](https://bradleyboehmke.github.io/HOML/08-bagging_files/figure-html/bagging-multiple-models-1.png)
*(Source: Bradley Boehmke - Bagging Architecture)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingClassifier, BaggingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, r2_score

np.random.seed(42)
print("Libraries imported.")

## 2. Understanding Bootstrapping

Bootstrapping stands for **sampling with replacement**. This means some samples from the original dataset will appear multiple times in a bootstrap sample, while others may be left out.

Let's visualize this process manually.

In [ ]:
# Artificial small dataset
original_data = np.array(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'])
n_samples = len(original_data)

print(f"Original Data: {original_data}")
print("-" * 30)

# Create 3 Bootstrap Samples
for i in range(1, 4):
    # Sample WITH replacement (replace=True is default)
    indices = np.random.choice(n_samples, size=n_samples, replace=True)
    bootstrap_sample = original_data[indices]
    
    # Identify Out-of-Bag (OOB) samples (those not in the bootstrap sample)
    unique_bootstrap = np.unique(bootstrap_sample)
    oob_sample = np.setdiff1d(original_data, unique_bootstrap)
    
    print(f"Bootstrap Sample {i}: {bootstrap_sample}")
    print(f"  -> Unique items in sample: {len(unique_bootstrap)}")
    print(f"  -> Out-of-Bag (OOB) items: {oob_sample}")
    print("-" * 30)

## 3. Out-of-Bag (OOB) Evaluation

On average, for each bootstrap sample, about **63.2%** of the training instances are sampled, and the remaining **36.8%** are not. These left-out instances are called **Out-of-Bag (OOB)** instances.

Since the model never sees the OOB instances during training, they can be used as a validation set to evaluate the performance without needing a separate hold-out set.

![OOB Error](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c7/Random_forest_error_sketch.png/800px-Random_forest_error_sketch.png)
*(Source: Wikipedia - Concept of OOB Error, acting as validation)*

*Note: The diagram illustrates how error stabilizes as trees increase, derived from OOB estimates.*

## 4. Bagging Classifier Implementation

Let's use `sklearn.ensemble.BaggingClassifier` on the Breast Cancer dataset. We will compare a single decision tree with a Bagging ensemble.

In [ ]:
# Load Dataset
df_cancer = pd.read_csv('../../00-data/wisconsin_breat_cancer.csv')
if 'diagnosis' in df_cancer.columns:
    df_cancer['diagnosis'] = df_cancer['diagnosis'].map({'M': 1, 'B': 0})
    X = df_cancer.drop(['id', 'diagnosis', 'Unnamed: 32'], axis=1, errors='ignore')
    y = df_cancer['diagnosis']
else:
    X = df_cancer.iloc[:, 2:]
    y = df_cancer.iloc[:, 1].map({'M': 1, 'B': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. train a Single Decision Tree
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
y_pred_tree = single_tree.predict(X_test)
print(f"Single Tree Test Accuracy: {accuracy_score(y_test, y_pred_tree):.4f}")

# 2. Train a Bagging Classifier (Ensemble of 100 Trees)
# oob_score=True allows us to see the OOB validation score
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=1.0, # Use all samples (with replacement)
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1 # Use all CPU cores
)
bag_clf.fit(X_train, y_train)

# Evaluate
y_pred_bag = bag_clf.predict(X_test)
print(f"Bagging OOB Score: {bag_clf.oob_score_:.4f}")
print(f"Bagging Test Accuracy: {accuracy_score(y_test, y_pred_bag):.4f}")

Usually, the Bagging Classifier performs slightly better or more reliably than a single Decision Tree because it has lower variance.

## 5. Random Forest: The Most Popular Bagging Method

**Random Forest** is an extension of Bagging specifically designed for Decision Trees, and it's one of the most widely used machine learning algorithms.

### What Makes Random Forest Special?

Random Forest adds an extra layer of randomness on top of Bagging:

1. **Bootstrap Sampling** (same as Bagging): Each tree is trained on a random bootstrap sample.
2. **Random Feature Selection** (new!): At each split, only a random subset of features is considered.

This "double randomness" makes trees even more diverse, leading to better ensemble performance.

### Key Differences: BaggingClassifier vs RandomForestClassifier

| Aspect | BaggingClassifier | RandomForestClassifier |
|--------|-------------------|------------------------|
| Base estimator | Any model | Decision Trees only |
| Feature selection | Uses all features | Random subset at each split |
| Diversity | From bootstrap samples | Bootstrap + feature randomness |
| Speed | Depends on base model | Optimized for trees |
| Feature importance | Not built-in | Built-in |

### When to Use Random Forest

- **General-purpose classification/regression**: Works well out-of-the-box
- **High-dimensional data**: Feature randomness helps with many features
- **Need feature importance**: Built-in feature ranking
- **Robust performance**: Less sensitive to hyperparameters than single trees

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest on the same breast cancer data
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',  # Random subset of features at each split
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)

# Evaluate
y_pred_rf = rf_clf.predict(X_test)
print(f"Random Forest OOB Score: {rf_clf.oob_score_:.4f}")
print(f"Random Forest Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")

# Compare all three models
print("\n" + "="*50)
print("COMPARISON: Single Tree vs Bagging vs Random Forest")
print("="*50)
print(f"Single Decision Tree:  {accuracy_score(y_test, y_pred_tree):.4f}")
print(f"Bagging Classifier:    {accuracy_score(y_test, y_pred_bag):.4f}")
print(f"Random Forest:         {accuracy_score(y_test, y_pred_rf):.4f}")

### Feature Importance with Random Forest

One of the most useful features of Random Forest is the built-in **feature importance** calculation. This tells us which features contribute most to the predictions.

In [ ]:
# Get feature importances from Random Forest
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_clf.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(10)
plt.barh(top_features['feature'], top_features['importance'], color='forestgreen')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()  # Highest importance at top
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(feature_importance.head(5).to_string(index=False))

## 6. Exercise: Bagging and Random Forest Regressor

In this exercise, you will apply Bagging and Random Forest to a regression problem: predicting `mpg`.

In [ ]:
# Load Data
df_mpg = pd.read_csv('../../00-data/auto_mpg.csv')
df_mpg = df_mpg.replace('?', np.nan).dropna()
X_reg = pd.get_dummies(df_mpg.drop(columns=['mpg', 'car name'], errors='ignore'), drop_first=True, dtype=float)   # this copy's origin is text
y_reg = df_mpg['mpg']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# TODO: Train a Single DecisionTreeRegressor and evaluate R2 score

# TODO: Train a BaggingRegressor with 100 trees (enable oob_score) and evaluate R2 score

# TODO: Train a RandomForestRegressor with 100 trees and evaluate R2 score
# Hint: from sklearn.ensemble import RandomForestRegressor

# Compare the results - which model performs best?